# Day 5 — ILT 3: Performance Levers
### GlobalMart Data Engineering · 2:00 PM – 3:00 PM

---

## Session Objectives

By the end of this session you will be able to:
- Identify the small file problem and its symptoms
- Run OPTIMIZE and explain what it does
- Configure AUTO OPTIMIZE to compact files automatically
- Explain the difference between Spark cache and Databricks disk cache
- Decide when to cache a DataFrame and when not to
- Compare Z-ORDER, Liquid Clustering, and broadcast joins as performance tools
- Run VACUUM safely and explain its purpose

---

## Agenda

| Time | Topic |
|------|-------|
| 2:00 | The small file problem — symptoms and causes |
| 2:10 | OPTIMIZE — compaction |
| 2:20 | AUTO OPTIMIZE — automatic compaction |
| 2:30 | Caching — Spark vs Databricks disk cache |
| 2:40 | Broadcast joins |
| 2:50 | VACUUM — safe deletion of old files |
| 2:55 | Decision framework + Q&A |

---
## 1. The Small File Problem

### How Small Files Are Created

```
Streaming writes (every 30 seconds):
  Trigger 1: 500 rows → 1 small Parquet file (~50 KB)
  Trigger 2: 500 rows → 1 small Parquet file (~50 KB)
  ...
  Trigger 1440: 500 rows → 1 small Parquet file (~50 KB)

After 24 hours of streaming:
  1,440 files × 50 KB = 72 MB total data
  But: 1,440 ADLS open() calls for any full scan
       1,440 Spark tasks minimum
       1,440 entries in Delta transaction log
```

### Symptoms of Small Files

| Symptom | What You See |
|---------|-------------|
| Slow queries despite small data | A 100 MB table takes 30 seconds to scan |
| High task count | Spark shows 1,000+ tasks for a simple COUNT |
| Slow table open | `spark.read.format('delta').load(path)` takes a long time just to start |
| Delta log bloat | `_delta_log/` has thousands of JSON files |

### Why ADLS Hates Small Files

```
ADLS (Azure Data Lake Storage) is optimised for:
  ✅ Sequential reads of large files (128 MB – several GB)
  ✅ High throughput per file

ADLS is slow for:
  ❌ Many small files — each open() has network round-trip latency
  ❌ Listing thousands of files — metadata operations are expensive

Ideal file size: 128 MB – 1 GB per file
Below 32 MB:     starting to become a problem
Below 1 MB:      definitely a problem
```

---
## 2. OPTIMIZE — Manual Compaction

OPTIMIZE reads many small files and rewrites them into fewer, larger files.

```
BEFORE OPTIMIZE:
  1,440 files × 50 KB = 72 MB

AFTER OPTIMIZE:
  1 file × 72 MB
  (or a few files if parallelism writes multiple)
```

### Running OPTIMIZE

```sql
-- Basic compaction:
OPTIMIZE delta.`abfss://amazon-data@amazonprojectadls.dfs.core.windows.net/bronze/adls/orders`;

-- Compaction + Z-ORDER (co-locate by column for faster filtering):
OPTIMIZE delta.`abfss://amazon-data@amazonprojectadls.dfs.core.windows.net/bronze/adls/orders`
ZORDER BY (CustomerID, OrderDate);

-- Compact only a specific partition (faster for large tables):
OPTIMIZE delta.`abfss://amazon-data@amazonprojectadls.dfs.core.windows.net/bronze/adls/orders`
WHERE ingestion_date = '2026-06-15';
```

```python
# Python equivalent:
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, bronze_path)
dt.optimize().executeCompaction()

# With Z-ORDER:
dt.optimize().executeZOrderBy(["CustomerID", "OrderDate"])
```

### What OPTIMIZE Does NOT Do

```
OPTIMIZE:
  ✅ Rewrites small files into large files
  ✅ Updates Delta transaction log (new files added, old files marked removed)
  ✅ Optionally Z-ORDERs data within files

OPTIMIZE does NOT:
  ❌ Delete the old small files (they stay, just marked as removed in log)
  ❌ Free up ADLS storage (need VACUUM for that — see later)
  ❌ Change the data — same rows, just reorganised into larger files
```

### When to Run OPTIMIZE

```
✅ After every large batch load
✅ Daily on streaming Bronze tables (schedule in Databricks Workflows)
✅ After a bulk DELETE or UPDATE on Silver/Gold
✅ Before a big analytical query that will run repeatedly

❌ Not necessary if using AUTO OPTIMIZE (below)
❌ Not on tables that are constantly being appended to mid-query
```

---
## 3. AUTO OPTIMIZE — Automatic Compaction

Instead of manually running OPTIMIZE, you can tell Delta to compact files automatically during writes.

### Two Settings

```python
# Option 1: Set on the Spark session (affects all Delta writes in this session)
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled",  "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled",    "true")

# Option 2: Set as Delta table properties (persists — affects all future writes)
spark.sql(f"""
    ALTER TABLE delta.`{bronze_path}`
    SET TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact'   = 'true'
    )
""")
```

### What Each Setting Does

| Setting | What It Does |
|---------|-------------|
| `optimizeWrite` | Before writing, Spark shuffles data to reduce the number of output files. Instead of 200 tasks × 1 file each = 200 small files, it targets ~128 MB per output file. |
| `autoCompact` | After a write completes, Delta runs a background OPTIMIZE on the written files. Happens asynchronously — doesn't slow down the write. |

### AUTO OPTIMIZE vs Manual OPTIMIZE

```
autoOptimize.optimizeWrite:
  → Prevents small files from being created in the first place
  → Happens AT write time
  → Slightly increases write time (shuffle overhead)

autoOptimize.autoCompact:
  → Compacts files AFTER they are written
  → Background operation
  → Does not include Z-ORDER (for Z-ORDER, still need manual OPTIMIZE)
```

> **Recommendation for Bronze streaming tables:** Enable both AUTO OPTIMIZE settings.
> For Z-ORDER, still schedule a daily manual OPTIMIZE ... ZORDER BY job.

---
## 4. Caching

Caching stores a DataFrame's data in memory (or on disk) so subsequent operations re-read from cache instead of re-scanning ADLS.

### Spark Cache (In-Memory)

```python
# Cache a DataFrame in memory:
bronze_df = spark.read.format("delta").load(bronze_path)
bronze_df.cache()

# First access — reads from ADLS, stores in memory:
bronze_df.count()  # triggers caching

# Second access — reads from memory:
bronze_df.count()   # much faster
bronze_df.filter(...).show()  # also from cache

# Free memory when done:
bronze_df.unpersist()
```

### Storage Levels

```python
from pyspark import StorageLevel

# Memory only (default .cache()):
df.persist(StorageLevel.MEMORY_ONLY)

# Memory + disk (spills to disk if memory full):
df.persist(StorageLevel.MEMORY_AND_DISK)

# Disk only (useful for DataFrames too large for memory):
df.persist(StorageLevel.DISK_ONLY)
```

### Databricks Disk Cache (Automatic)

```
Databricks has a second layer of caching called the "Disk Cache" (formerly Delta Cache):

- Operates at the STORAGE LAYER — caches raw Parquet data from ADLS
- Enabled automatically on some cluster types (General Purpose, Storage Optimized)
- Transparent — no code changes needed
- Persists ACROSS Spark jobs on the same cluster
- Survives notebook detach/reattach (unlike Spark cache)

Spark cache vs Databricks disk cache:
  Spark cache:     you control it (.cache(), .unpersist())
                   stored as Spark RDD partitions in JVM heap
                   lost when cluster restarts

  Databricks cache: automatic
                    stored on local SSDs of cluster nodes
                    survives across jobs, faster for large scans
```

### When to Cache, When Not To

| Scenario | Cache? | Reason |
|----------|--------|--------|
| Small reference table used in multiple joins | ✅ Yes | Avoids re-reading from ADLS on each join |
| Large Bronze table for a single scan | ❌ No | Cache won't fit in memory; single scan doesn't benefit |
| ML feature table iterated 100 times | ✅ Yes | Massive saving — 100 ADLS reads → 1 ADLS read |
| Streaming source | ❌ No | Streaming state managed differently |
| Iterative aggregations on the same DF | ✅ Yes | Each `groupBy` would re-read from disk without cache |

---
## 5. Broadcast Joins

When joining a large table to a small table, tell Spark to broadcast the small table to all workers instead of doing a sort-merge join.

### Sort-Merge Join (Default — Expensive)

```
Large table: bronze_orders (1.4M rows)
Small table: shipping_tiers (5 rows)

Default sort-merge join:
  1. Shuffle ALL of bronze_orders across network (1.4M rows moved)
  2. Shuffle ALL of shipping_tiers across network (5 rows moved)
  3. Sort both sides
  4. Merge

Problem: shuffling 1.4M rows across the network is expensive
```

### Broadcast Join (Small Table → All Workers)

```python
from pyspark.sql.functions import broadcast

# Option 1: Explicit broadcast hint:
result = bronze_orders.join(
    broadcast(shipping_tiers),
    on="ShippingTierID",
    how="left"
)

# Option 2: Let Spark decide automatically (if table < autoBroadcastJoinThreshold):
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "50mb")  # default 10MB
```

```
Broadcast join:
  1. Ship entire shipping_tiers (5 rows) to ALL worker nodes once
  2. Each worker joins its local partition of bronze_orders to its local copy
  3. No network shuffle of bronze_orders at all

Result: 100x+ faster for small-to-large joins
```

### When to Use Broadcast

```
Use broadcast when:
  ✅ One table is small (< 100 MB — adjust threshold if needed)
  ✅ The small table is used repeatedly (broadcast once, reuse)
  ✅ Joining lookup/reference tables: shipping_tiers, products, suppliers

Do NOT broadcast when:
  ❌ Both tables are large (both need to shuffle → no benefit)
  ❌ Table is > 1 GB (driver runs out of memory collecting it)
```

---
## 6. VACUUM — Cleaning Up Old Files

Recall: OPTIMIZE marks old small files as "removed" in the Delta log but does NOT delete them from ADLS.

VACUUM physically deletes those removed files to free storage.

### Running VACUUM

```sql
-- Delete files older than 7 days (default):
VACUUM delta.`abfss://amazon-data@amazonprojectadls.dfs.core.windows.net/bronze/adls/orders`;

-- Custom retention (minimum 7 days recommended):
VACUUM delta.`abfss://...` RETAIN 168 HOURS;  -- 7 days

-- Dry run (shows what WOULD be deleted without actually deleting):
VACUUM delta.`abfss://...` RETAIN 168 HOURS DRY RUN;
```

### The 7-Day Retention Rule

```
Default retention: 7 days (168 hours)

Why 7 days minimum?
  Delta time travel works by reading old file versions from the transaction log.
  If you VACUUM files that are still referenced by recent versions,
  time travel to those versions will fail with FileNotFoundException.

  7 days = safe window for most use cases

Dangerous:
  VACUUM delta.`...` RETAIN 0 HOURS;
  → Deletes ALL previous version files immediately
  → Time travel is permanently disabled for this table
  → Cannot be undone
```

### VACUUM vs OPTIMIZE — Summary

```
OPTIMIZE:   Compacts files (rewrites small → large)
             Old files marked removed in log
             ADLS storage NOT freed

VACUUM:     Deletes files marked as removed by OPTIMIZE (or DELETE/UPDATE)
            ADLS storage IS freed
            Time travel to deleted versions is no longer possible

Typical maintenance schedule:
  Daily:   OPTIMIZE (compaction + Z-ORDER)
  Weekly:  VACUUM RETAIN 168 HOURS (free storage, keep 7-day time travel)
```

---
## 7. Performance Lever Decision Framework

```
PERFORMANCE PROBLEM?
      |
      ├── Query is slow on a large table
      │       |
      │       ├── Many small files?  → OPTIMIZE (compact) + AUTO OPTIMIZE (prevent)
      │       ├── Filtering by a column?  → ZORDER BY that column
      │       ├── Joining small table?  → broadcast() the small table
      │       └── Scanning same data repeatedly?  → .cache() the DataFrame
      │
      ├── Storage cost is too high
      │       → VACUUM RETAIN 168 HOURS (delete old file versions)
      │
      ├── Write is slow (many small output files)
      │       → Enable optimizeWrite = true
      │
      └── Cluster startup is slow
              → Use Databricks disk cache (cluster type: Storage Optimized)
              → Pre-warm cluster by running a dummy scan at startup
```

---

## Key Takeaways

1. **Small files** = the #1 performance issue in streaming Bronze pipelines — OPTIMIZE fixes it
2. **AUTO OPTIMIZE** = enable on all streaming Delta tables to prevent small files
3. **Cache** small reference tables and DataFrames used in multiple operations — not large tables
4. **Broadcast** the small side of every join under ~100 MB
5. **VACUUM** weekly to free ADLS storage — always keep 7-day retention minimum
6. **Z-ORDER** and **Liquid Clustering** improve query speed by co-locating related data

---

## Discussion Questions

1. *A Bronze table has 10,000 files averaging 30 KB each. What command fixes this? What does it actually do to the data?*

2. *After running OPTIMIZE, your ADLS storage usage hasn't gone down. Why? What command do you need?*

3. *A data scientist runs the same 200 MB ML feature table through 50 different model training iterations in one notebook. How should you help them speed this up?*

4. *You try to time travel to a Delta table version from 10 days ago, but get a FileNotFoundException. What most likely happened?*

5. *Your Silver job joins bronze_orders (500M rows) with dim_products (2,000 rows). What join optimisation should you apply and why?*